# API Tools

For this section we'll just get a default ReAct agent to run our tools; this will let us focus our debugging in the tools themselves rather than the agent.

In [74]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

**EDIT: Moved all API handling code to [`football_api_utils.py`](football_api_utils.py)** so it can be used in other notebooks as well.

In [3]:
!uv pip install python-dotenv requests pydantic

# Import the Football API utility functions from our module
from football_api_utils import (
    Response, 
    ValidResponse, 
    ErrorResponse, 
    check_api_response_status, 
    call_football_api
)

Audited 2 packages in 28ms
Audited 1 package in 5ms
Audited 1 package in 5ms


In [33]:
# Test the API functions
print("Testing API status...")
status_response = call_football_api("GET", "status")
print(f"Status response: {status_response}")

print("\nTesting error handling with invalid endpoint...")
error_response = call_football_api("GET", "invalid_endpoint")
print(f"Error response: {error_response}")

Testing API status...
Status response: data={'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Vasco', 'lastname': 'Peleteiro', 'email': 'vasco@augustalabs.ai'}, 'subscription': {'plan': 'Pro', 'end': '2025-07-05T20:58:08+00:00', 'active': True}, 'requests': {'current': 30, 'limit_day': 7500}}}

Testing error handling with invalid endpoint...
Error response: data={'get': 'invalid_endpoint', 'parameters': [], 'errors': {'endpoint': 'The Invalid_endpoint endpoint does not exist.'}, 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': []}
Status response: data={'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Vasco', 'lastname': 'Peleteiro', 'email': 'vasco@augustalabs.ai'}, 'subscription': {'plan': 'Pro', 'end': '2025-07-05T20:58:08+00:00', 'active': True}, 'requests': {'current': 30, 'limit_day': 7

We need to implement a tool that can query the [Football API](https://www.api-football.com/documentation-v3) to get information about football leagues, teams, players, and matches.

1. Classificações de equipas — <https://www.api-football.com/documentation-v3#tag/Standings/operation/get-standings>
2. Próximos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `next` parameter
3. Últimos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `last` parameter
4. Jogos específicos (ex: SLB vs SCP para a liga em 2012/13) — we need to get the fixture ID first, then use it to get the match details.
5. Resultados de jogos específicos
6. Eventos de jogos específicos (ex.: golos, cartões, substituições).
7. Estatísticas de jogadores (ex: número de golos do jogador X na época Y)

Qualquer equipa, jogo ou jogador das top 7 ligas europeias + das 3 competições europeias.

Extra:

8. Odds
9. H2H
10. ...


## API Football

There doesn't seem to be a OpenAPI spec for this API, but we'll use the documentation to implement the tools we need and see how it goes.

### 0. Setup

**EDIT: Moved cache generation (`top_leagues.json` and `top_teams.json`)  to [`03a_tools_setup.ipynb`](03a_tools_setup.ipynb) to avoid clutter and repeated execution of costly API calls.**

In [91]:
import json

# Define function to get league ID by name
def get_league_id_by_name(league_name: str) -> int | None:
    """
    Get the league ID by its name.

    Args:
        league_name: Name of the league

    Returns:
        League ID if found, otherwise None
    """
    try:
        leagues = json.load(open("top_leagues.json"))
    except FileNotFoundError:
        logger.warning("top_leagues.json not found. Will fetch from API...")
        leagues = {}

    # Exact match first
    if league_name in leagues:
        return leagues[league_name]

    # If the league is not found, partial match the name
    #for league_key in leagues:
    #    if league_name.lower() in league_key.lower() or league_key.lower() in league_name.lower():
    #        return leagues[league_key]

    # If no match is found, fetch it from the API:
    logger.warning(
        f"League '{league_name}' not found in selected leagues. Fetching from API..."
    )
    response = call_football_api("GET", "leagues", params={"search": league_name})
    if isinstance(response, ValidResponse):
        # Work directly with the API response data
        if 'response' in response.data:
            for item in response.data['response']:
                league = item.get('league', {})
                league_api_name = league.get('name', '')
                league_id = league.get('id')
                
                # Check if this league matches what we're looking for
                if (league_api_name.lower() in league_name.lower() or 
                    league_name.lower() in league_api_name.lower()):
                    return league_id
        
        logger.error(f"League '{league_name}' not found in API response.")

    return None


def list_leagues() -> list[str]:
    """
    List all available leagues.

    Returns:
        List of league names
    """
    try:
        leagues = json.load(open("top_leagues.json"))
        return list(leagues.keys())
    except FileNotFoundError:
        logger.warning("top_leagues.json not found.")
        return []

In [12]:
# Test exact league match
league_name = "Premier League"
league_id = get_league_id_by_name(league_name)
print(f"League: {league_name}, ID: {league_id}")

League: Premier League, ID: 39


In [13]:
# Test partial match
league_name_partial = "Premier"
league_id_partial = get_league_id_by_name(league_name_partial)
print(f"League (partial): {league_name_partial}, ID: {league_id_partial}")

League (partial): Premier, ID: 39


In [14]:
# Test list leagues function
available_leagues = list_leagues()
print(f"Available leagues: {available_leagues[:5]}...")  # Show first 5

Available leagues: ['Ligue 1', 'Premier League', 'Bundesliga', 'Serie A', 'Eredivisie']...


In [15]:
# Test with a league not in the list
league_name_not_found = "Nonexistent League"
league_id_not_found = get_league_id_by_name(league_name_not_found)
print(f"League (not found): {league_name_not_found}, ID: {league_id_not_found}")

ERROR:__main__:League 'Nonexistent League' not found in API response.
ERROR:__main__:League 'Nonexistent League' not found in API response.


League (not found): Nonexistent League, ID: None


In [16]:
# Test with a league not in the list that exists in the API
league_name_exists = "Jupiler Pro League"
league_id_exists = get_league_id_by_name(league_name_exists)
print(f"League (exists in API): {league_name_exists}, ID: {league_id_exists}")

League (exists in API): Jupiler Pro League, ID: 144


In [75]:
def get_team_id_by_name(team_name: str, league_name: str | None = None) -> int | None:
    """
    Get the team ID by its name.

    Args:
        team_name: Name of the team
        league_name: Optional league name to filter teams by league

    Returns:
        Team ID if found, otherwise None
    """
    try:
        teams = json.load(open("top_teams.json"))
    except FileNotFoundError:
        logger.warning("top_teams.json not found. Will fetch from API...")
        teams = {}

    # Exact match first
    if team_name in teams:
        team_data = teams[team_name]
        # If league filter is specified, check if team participates in that league
        if league_name:
            team_leagues = team_data.get("leagues", [])
            if league_name not in team_leagues:
                logger.warning(f"Team '{team_name}' found but does not participate in '{league_name}'")
                # Continue to partial match or API search
            else:
                return team_data["id"]
        else:
            return team_data["id"]

    # If the team is not found, try partial match
    for team_key, team_data in teams.items():
        if team_name.lower() in team_key.lower() or team_key.lower() in team_name.lower():
            # If league filter is specified, check if team participates in that league
            if league_name:
                team_leagues = team_data.get("leagues", [])
                if league_name not in team_leagues:
                    continue
            return team_data["id"]

    # If no match is found, fetch it from the API:
    filter_msg = f" in league '{league_name}'" if league_name else ""
    logger.warning(
        f"Team '{team_name}'{filter_msg} not found in cached teams. Fetching from API..."
    )
    response = call_football_api("GET", "teams", params={"search": team_name})

    if isinstance(response, ValidResponse):
        for team_data in response.data["response"]:
            team = team_data["team"]
            if team["name"].lower() == team_name.lower():
                return team["id"]

    return None


def search_teams_by_code(team_code: str, league_name: str | None = None) -> list[dict]:
    """
    Search for all teams with a specific code.

    Args:
        team_code: Code of the team (e.g., "MUN", "LIV")
        league_name: Optional league name to filter teams by league

    Returns:
        List of team dictionaries with name, id, code, and leagues
    """
    try:
        teams = json.load(open("top_teams.json"))
    except FileNotFoundError:
        logger.warning("top_teams.json not found.")
        return []

    matching_teams = []
    for team_name, team_data in teams.items():
        if team_data.get("code") and team_data["code"].upper() == team_code.upper():
            # If league filter is specified, check if team participates in that league
            if league_name:
                team_leagues = team_data.get("leagues", [])
                if league_name not in team_leagues:
                    continue
            
            team_info = {
                "name": team_name,
                "id": team_data["id"],
                "code": team_data["code"],
                "leagues": team_data.get("leagues", [])
            }
            matching_teams.append(team_info)

    return matching_teams


def list_teams(league_name: str | None = None) -> list[str]:
    """
    List all available teams from cached data.

    Args:
        league_name: Optional league name to filter teams by league

    Returns:
        List of team names
    """
    try:
        teams = json.load(open("top_teams.json"))
        if league_name:
            # Filter teams by league if specified
            filtered_teams = [
                team_name for team_name, team_data in teams.items()
                if league_name in team_data.get("leagues", [])
            ]
            return filtered_teams
        return list(teams.keys())
    except FileNotFoundError:
        logger.warning("top_teams.json not found.")
        return []



In [85]:
 # Test the new team lookup functions

# Test exact team name match
team_name = "Manchester United"
team_id = get_team_id_by_name(team_name)
print(f"Team: {team_name}, ID: {team_id}")

# Test team name match with league filter
team_id_filtered = get_team_id_by_name("Manchester United", "Premier League")
print(f"Team 'Manchester United' in Premier League, ID: {team_id_filtered}")

# Test team code search (all teams with same code)
team_code_duplicate = "SLB"
matching_teams = search_teams_by_code(team_code_duplicate)
print(f"Teams with code '{team_code_duplicate}':")
for team in matching_teams:
    print(f"  - {team['name']} (ID: {team['id']}, Leagues: {team['leagues']})")

# Test team code search with league filter
matching_teams_filtered = search_teams_by_code("MAN", "Premier League")
print(f"Teams with code 'MAN' in Premier League:")
for team in matching_teams_filtered:
    print(f"  - {team['name']} (ID: {team['id']}, Leagues: {team['leagues']})")

# Test list teams function
available_teams = list_teams()
print(f"Total teams available: {len(available_teams)}")
print(f"First 5 teams: {available_teams[:5]}")  # Show first 5

Team: Manchester United, ID: 33
Team 'Manchester United' in Premier League, ID: 33
Teams with code 'SLB':
Teams with code 'MAN' in Premier League:
Total teams available: 1622
First 5 teams: ['Angers', 'Lille', 'Lyon', 'Marseille', 'Montpellier']


In [19]:
# Test exact team name match
team_name = "Manchester United"
team_id = get_team_id_by_name(team_name)
print(f"Team: {team_name}, ID: {team_id}")

Team: Manchester United, ID: 33


In [20]:
# Test partial team name match
team_name_partial = "Arsenal"
team_id_partial = get_team_id_by_name(team_name_partial)
print(f"Team (partial): {team_name_partial}, ID: {team_id_partial}")

Team (partial): Arsenal, ID: 42


In [21]:
# Test with a team not in the list
team_name_not_found = "Nonexistent Team"
team_id_not_found = get_team_id_by_name(team_name_not_found)
print(f"Team (not found): {team_name_not_found}, ID: {team_id_not_found}")

Team (not found): Nonexistent Team, ID: None


In [22]:
# Test with a team that exists in the API but not in the cached data
team_name_exists = "Fluminense"
team_id_exists = get_team_id_by_name(team_name_exists)
print(f"Team (exists in API): {team_name_exists}, ID: {team_id_exists}")

Team (exists in API): Fluminense, ID: 124


In [23]:
# Test list teams function
available_teams = list_teams()
print(f"Total teams available: {len(available_teams)}")
print(f"First 5 teams: {available_teams[:5]}")  # Show first 5

Total teams available: 1622
First 5 teams: ['Angers', 'Lille', 'Lyon', 'Marseille', 'Montpellier']


In [78]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from typing import Optional

# Create tools for helper functions
class ListLeaguesInput(BaseModel):
    pass  # No parameters needed

class ListTeamsInput(BaseModel):
    league_name: Optional[str] = Field(None, description="Optional league name to filter teams by league")

class SearchTeamsByCodeInput(BaseModel):
    team_code: str = Field(..., description="Code of the team (e.g., 'MUN', 'LIV', 'SLB')")
    league_name: Optional[str] = Field(None, description="Optional league name to filter teams by league")

# Create the tools
list_leagues_tool = StructuredTool.from_function(
    list_leagues,
    name="list_leagues",
    description="List all available leagues from cached data.",
    args_schema=ListLeaguesInput,
    return_direct=False
)

list_teams_tool = StructuredTool.from_function(
    list_teams,
    name="list_teams",
    description="List all available teams, optionally filtered by league.",
    args_schema=ListTeamsInput,
    return_direct=False
)

search_teams_by_code_tool = StructuredTool.from_function(
    search_teams_by_code,
    name="search_teams_by_code",
    description="Search for teams by their code (e.g., 'MUN' for Manchester United, 'SLB' for Benfica).",
    args_schema=SearchTeamsByCodeInput,
    return_direct=False
)

In [79]:
!uv pip install -qU "langchain[google-genai]"

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [81]:
from langgraph.prebuilt import create_react_agent

# Test the helper tools with a ReAct agent
helper_tools = [list_leagues_tool, list_teams_tool, search_teams_by_code_tool]

react_agent_helper = create_react_agent(
    model=model,
    tools=helper_tools,
    prompt="You are a helpful assistant that can help users explore available leagues and teams."
)

print("Testing helper tools:")
print("\n1. Testing list_leagues:")
result1 = react_agent_helper.invoke(
    {"messages": [{"role": "user", "content": "What leagues are available?"}]}
)
print(result1)

print("\n2. Testing search_teams_by_code:")
result2 = react_agent_helper.invoke(
    {"messages": [{"role": "user", "content": "Find all teams with code 'MAN'"}]}
)
print(result2)

print("\n3. Testing list_teams with league filter:")
result3 = react_agent_helper.invoke(
    {"messages": [{"role": "user", "content": "List teams in the Premier League"}]}
)
print(result3)

Testing helper tools:

1. Testing list_leagues:
{'messages': [HumanMessage(content='What leagues are available?', additional_kwargs={}, response_metadata={}, id='1b604c09-3de2-4bb4-b09e-6ef5ea52c2d8'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'list_leagues', 'arguments': '{}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--2b994a98-c001-416f-ab0a-929c8ef44d1d-0', tool_calls=[{'name': 'list_leagues', 'args': {}, 'id': '7c5a67e5-5677-4d02-b75d-4b11138bbda4', 'type': 'tool_call'}], usage_metadata={'input_tokens': 131, 'output_tokens': 4, 'total_tokens': 135, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='["Ligue 1", "Premier League", "Bundesliga", "Serie A", "Eredivisie", "Primeira Liga", "La Liga", "UEFA Champions League", "Coupe de France", "FA Cup", "DFB Pokal", "UEFA Europa League", "UEFA Super Cup", "Taça de Portuga

## 1. Team standings

<https://www.api-football.com/documentation-v3#tag/Standings/operation/get-standings>

In [24]:
def get_standings(league_name: str, season: int, team_name: str | None = None) -> Response:
    """
    Get the standings for a league or specific team in a league.
    
    Args:
        league_name: Name of the league (e.g., "Premier League", "La Liga")
        season: Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)
        team_name: Optional team name to filter standings for specific team
        
    Returns:
        ValidResponse with standings data or ErrorResponse with error details
    """
    # Get league ID from the league name
    league_id = get_league_id_by_name(league_name)
    if league_id is None:
        logger.error(f"League '{league_name}' not found")
        return ErrorResponse(error=f"League '{league_name}' not found")
    
    # Prepare parameters for the API call
    params = {
        "league": league_id,
        "season": season
    }
    
    # If team name is provided, get team ID and add to params
    if team_name:
        team_id = get_team_id_by_name(team_name)
        if team_id is None:
            logger.error(f"Team '{team_name}' not found")
            return ErrorResponse(error=f"Team '{team_name}' not found")
        params["team"] = team_id
    
    logger.info(f"Fetching standings for {league_name} (ID: {league_id}) season {season}")
    if team_name:
        logger.info(f"Filtering for team: {team_name}")
    
    # Make the API call
    try:
        response = call_football_api("GET", "standings", params=params)
        
        if isinstance(response, ValidResponse) and "response" in response.data and response.data["response"]:
            logger.info(f"Successfully retrieved standings data")
            return response
        elif isinstance(response, ValidResponse):
            logger.warning(f"No standings data found for {league_name} season {season}")
            return ErrorResponse(error=f"No standings data found for {league_name} season {season}")
        else:
            return response  # Already an ErrorResponse
            
    except Exception as e:
        logger.error(f"Error fetching standings: {str(e)}")
        return ErrorResponse(error=f"Error fetching standings: {str(e)}")


def format_standings_table(standings_response: Response) -> str:
    """
    Format the standings response into a readable table string.
    
    Args:
        standings_response: Response from get_standings function
        
    Returns:
        Formatted string table of the standings
    """
    if not isinstance(standings_response, ValidResponse):
        return f"Error: {standings_response.error}" # type: ignore
    
    data = standings_response.data
    if "response" not in data or not data["response"]:
        return "No standings data available"
    
    # Extract standings data
    standings_data = data["response"][0]["league"]["standings"][0]
    
    # Create table header
    table = f"{'Pos':<4} {'Team':<25} {'GP':<3} {'W':<3} {'D':<3} {'L':<3} {'GF':<3} {'GA':<3} {'GD':<4} {'Pts':<4}\n"
    table += "-" * 75 + "\n"
    
    # Add each team's data
    for team_data in standings_data:
        rank = team_data["rank"]
        team_name = team_data["team"]["name"]
        all_stats = team_data["all"]
        
        played = all_stats["played"]
        wins = all_stats["win"]
        draws = all_stats["draw"]
        losses = all_stats["lose"]
        goals_for = all_stats["goals"]["for"]
        goals_against = all_stats["goals"]["against"]
        goal_diff = goals_for - goals_against
        points = team_data["points"]
        
        # Truncate team name if too long
        display_name = team_name[:24] if len(team_name) > 24 else team_name
        
        table += f"{rank:<4} {display_name:<25} {played:<3} {wins:<3} {draws:<3} {losses:<3} {goals_for:<3} {goals_against:<3} {goal_diff:<4} {points:<4}\n"
    
    return table


First we test the function:

In [25]:
test_standings = get_standings("Premier League", 2024)
print("Raw API Response:")
print(test_standings)
print("\n" + "="*50 + "\n")
print("Formatted Table:")
print(format_standings_table(test_standings))

INFO:__main__:Fetching standings for Premier League (ID: 39) season 2024
INFO:__main__:Successfully retrieved standings data
INFO:__main__:Successfully retrieved standings data


Raw API Response:
data={'get': 'standings', 'parameters': {'league': '39', 'season': '2024'}, 'errors': [], 'results': 1, 'paging': {'current': 1, 'total': 1}, 'response': [{'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2024, 'standings': [[{'rank': 1, 'team': {'id': 40, 'name': 'Liverpool', 'logo': 'https://media.api-sports.io/football/teams/40.png'}, 'points': 84, 'goalsDiff': 45, 'group': 'Premier League', 'form': 'DLDLW', 'status': 'same', 'description': 'Champions League', 'all': {'played': 38, 'win': 25, 'draw': 9, 'lose': 4, 'goals': {'for': 86, 'against': 41}}, 'home': {'played': 19, 'win': 14, 'draw': 4, 'lose': 1, 'goals': {'for': 42, 'against': 16}}, 'away': {'played': 19, 'win': 11, 'draw': 5, 'lose': 3, 'goals': {'for': 44, 'against': 25}}, 'update': '2025-05-26T00:00:00+00:00'}, {'rank': 2, 'team': {'id': 42, 'name': 'Arse

Then we convert it into a tool and create a ReAct agent to use it.

In [40]:
class GetStandingsInput(BaseModel):
    league_name: str = Field(..., description="Name of the league or cup (e.g., 'Premier League', 'La Liga')")
    season: int = Field(..., description="Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)")
    team_name: Optional[str] = Field(None, description="Optional team name to filter standings for specific team")

# create a tool from the get_standings function
get_standings_tool = StructuredTool.from_function(
    get_standings,
    name="get_standings",
    description="Get the standings for a league or specific team in a league.",
    args_schema=GetStandingsInput,
    return_direct=False
)

print(get_standings_tool.name)
print(get_standings_tool.description)
print(get_standings_tool.args)

get_standings
Get the standings for a league or specific team in a league.
{'league_name': {'description': "Name of the league (e.g., 'Premier League', 'La Liga')", 'title': 'League Name', 'type': 'string'}, 'season': {'description': 'Season year (4 digits, e.g., 2024)', 'title': 'Season', 'type': 'integer'}, 'team_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional team name to filter standings for specific team', 'title': 'Team Name'}}


In [41]:
# Create a ReAct agent to use the tool
react_agent = create_react_agent(
    model=model,
    tools=[get_standings_tool],
    prompt="You are a helpful assistant. The latest season is 2024 and we are in early 2025."
)

Let's test our agent with a simple query:

In [42]:
react_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current standing of Sporting in the Primeira Liga?"}]}
)

INFO:__main__:Fetching standings for Primeira Liga (ID: 94) season 2024
INFO:__main__:Filtering for team: Sporting
INFO:__main__:Filtering for team: Sporting
INFO:__main__:Successfully retrieved standings data
INFO:__main__:Successfully retrieved standings data


{'messages': [HumanMessage(content='what is the current standing of Sporting in the Primeira Liga?', additional_kwargs={}, response_metadata={}, id='3ac01fe9-18fa-45a4-a040-cf1fb66c9a92'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_standings', 'arguments': '{"league_name": "Primeira Liga", "season": 2024.0, "team_name": "Sporting"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--af52b4ee-a07c-43d5-927e-9079e5ef8708-0', tool_calls=[{'name': 'get_standings', 'args': {'league_name': 'Primeira Liga', 'season': 2024.0, 'team_name': 'Sporting'}, 'id': '50d1c123-aefc-421f-8670-ae41a899dbf0', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 16, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="data={'get': 'standings', 'parameters': {'league': '94', 'season': '2024', 'team

And now a fake query to see how it handles errors:

In [43]:
react_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current standing of Sporting in the Eredivisie?"}]}
)

INFO:__main__:Fetching standings for Eredivisie (ID: 88) season 2024
INFO:__main__:Filtering for team: Sporting
INFO:__main__:Filtering for team: Sporting


{'messages': [HumanMessage(content='what is the current standing of Sporting in the Eredivisie?', additional_kwargs={}, response_metadata={}, id='88fdfb95-cf4e-4577-9353-0d520275ac0a'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_standings', 'arguments': '{"league_name": "Eredivisie", "season": 2024.0, "team_name": "Sporting"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--e29ac489-2afa-43f5-8eb9-2cfb865a4f72-0', tool_calls=[{'name': 'get_standings', 'args': {'league_name': 'Eredivisie', 'season': 2024.0, 'team_name': 'Sporting'}, 'id': '86a1639f-3c45-40f1-832e-12fc0fe2699b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 16, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="error='No standings data found for Eredivisie season 2024' status_code=None", name='get_s

## 2. Upcoming fixtures

<https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `next` parameter

In [48]:
def get_next_fixtures(
    league_name: str | None = None,
    season: int | None = None,
    team_name: str | None = None,
    next_count: int = 5,
    date_from: str | None = None,
    date_to: str | None = None
) -> Response:
    """
    Get the next fixtures for a league, team, or date range.
    
    Args:
        league_name: Name of the league or cup (e.g., "Premier League", "La Liga")
        season: Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)

        team_name: Optional team name to filter fixtures for specific team
        next_count: Optional number of next fixtures to retrieve (max 20)
        date_from: Optional start date in YYYY-MM-DD format
        date_to: Optional end date in YYYY-MM-DD format
        
    Returns:
        ValidResponse with fixtures data or ErrorResponse with error details
    """
    params = {}
    
    # Add league parameter if provided
    if league_name:
        league_id = get_league_id_by_name(league_name)
        if league_id is None:
            logger.error(f"League '{league_name}' not found")
            return ErrorResponse(error=f"League '{league_name}' not found")
        params["league"] = league_id
    
    # Add season parameter if provided
    if season:
        params["season"] = season
    
    # Add team parameter if provided
    if team_name:
        team_id = get_team_id_by_name(team_name)
        if team_id is None:
            logger.error(f"Team '{team_name}' not found")
            return ErrorResponse(error=f"Team '{team_name}' not found")
        params["team"] = team_id
    
    # Add next parameter (number of next fixtures)
    if next_count and next_count > 0:
        params["next"] = min(next_count, 20)  # API limit is 20
    
    # Add date range parameters if provided
    if date_from:
        params["from"] = date_from
    if date_to:
        params["to"] = date_to
    
    logger.info(f"Fetching next {next_count} fixtures")
    if league_name:
        logger.info(f"League: {league_name}")
    if team_name:
        logger.info(f"Team: {team_name}")
    
    # Make the API call
    try:
        response = call_football_api("GET", "fixtures", params=params)
        
        if isinstance(response, ValidResponse) and "response" in response.data:
            logger.info(f"Successfully retrieved {len(response.data['response'])} fixtures")
            return response
        elif isinstance(response, ValidResponse):
            logger.warning(f"No fixtures found")
            return ErrorResponse(error="No fixtures found")
        else:
            return response  # Already an ErrorResponse
            
    except Exception as e:
        logger.error(f"Error fetching fixtures: {str(e)}")
        return ErrorResponse(error=f"Error fetching fixtures: {str(e)}")

In [49]:
# Test
fixtures = get_next_fixtures(league_name="Club World Cup", season=2025, team_name="Benfica", next_count=5)
print("Raw API Response:")
print(fixtures)

INFO:__main__:Fetching next 5 fixtures
INFO:__main__:League: Club World Cup
INFO:__main__:Team: Benfica
INFO:__main__:Successfully retrieved 1 fixtures


Raw API Response:
data={'get': 'fixtures', 'parameters': {'league': '15', 'season': '2025', 'team': '211', 'next': '5'}, 'errors': [], 'results': 1, 'paging': {'current': 1, 'total': 1}, 'response': [{'fixture': {'id': 1321716, 'referee': None, 'timezone': 'UTC', 'date': '2025-06-24T19:00:00+00:00', 'timestamp': 1750791600, 'periods': {'first': None, 'second': None}, 'venue': {'id': 18659, 'name': 'Bank of America Stadium', 'city': 'Charlotte, North Carolina'}, 'status': {'long': 'Not Started', 'short': 'NS', 'elapsed': None, 'extra': None}}, 'league': {'id': 15, 'name': 'FIFA Club World Cup', 'country': 'World', 'logo': 'https://media.api-sports.io/football/leagues/15.png', 'flag': None, 'season': 2025, 'round': 'Group Stage - 3', 'standings': True}, 'teams': {'home': {'id': 211, 'name': 'Benfica', 'logo': 'https://media.api-sports.io/football/teams/211.png', 'winner': None}, 'away': {'id': 157, 'name': 'Bayern München', 'logo': 'https://media.api-sports.io/football/teams/157.png', 'w

In [50]:
class GetNextFixturesInput(BaseModel):
    league_name: Optional[str] = Field(None, description="Name of the league or cup (e.g., 'Premier League', 'La Liga')")
    season: Optional[int] = Field(None, description="Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)")
    team_name: Optional[str] = Field(None, description="Optional team name to filter fixtures for specific team")
    next_count: int = Field(5, description="Number of next fixtures to retrieve (max 20)")
    date_from: Optional[str] = Field(None, description="Optional start date in YYYY-MM-DD format")
    date_to: Optional[str] = Field(None, description="Optional end date in YYYY-MM-DD format")

# create a tool from the get_next_fixtures function
get_next_fixtures_tool = StructuredTool.from_function(
    get_next_fixtures,
    name="get_next_fixtures",
    description="Get the next fixtures for a league, team, or date range.",
    args_schema=GetNextFixturesInput,
    return_direct=False
)

print(get_next_fixtures_tool.name)
print(get_next_fixtures_tool.description)
print(get_next_fixtures_tool.args)

get_next_fixtures
Get the next fixtures for a league, team, or date range.
{'league_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': "Name of the league or cup (e.g., 'Premier League', 'La Liga')", 'title': 'League Name'}, 'season': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Season year (4 digits, e.g., 2024)', 'title': 'Season'}, 'team_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional team name to filter fixtures for specific team', 'title': 'Team Name'}, 'next_count': {'default': 5, 'description': 'Number of next fixtures to retrieve (max 20)', 'title': 'Next Count', 'type': 'integer'}, 'date_from': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional start date in YYYY-MM-DD format', 'title': 'Date From'}, 'date_to': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional end date 

In [60]:
from datetime import datetime
react_agent = create_react_agent(
    model=model,
    tools=[get_next_fixtures_tool],
    prompt=f"You are a helpful assistant. Today is {datetime.today().strftime('%Y-%m-%d')}. If the user query does not specify a season, assume it is 2024 for European leagues and 2025 for cups."
)

react_agent.invoke(
    {"messages": [{"role": "user", "content": "what are the next fixtures for Borussia Dortmund in the Club World Cup?"}]}
)

INFO:__main__:Fetching next 5 fixtures
INFO:__main__:League: Club World Cup
INFO:__main__:Team: Borussia Dortmund
INFO:__main__:Fetching next 5 fixtures
INFO:__main__:League: Club World Cup
INFO:__main__:Team: Borussia Dortmund
INFO:__main__:Successfully retrieved 1 fixtures
INFO:__main__:Successfully retrieved 1 fixtures


{'messages': [HumanMessage(content='what are the next fixtures for Borussia Dortmund in the Club World Cup?', additional_kwargs={}, response_metadata={}, id='9fcfff4b-b882-4e25-8ae5-5a7afbe96bea'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_next_fixtures', 'arguments': '{"league_name": "Club World Cup", "season": 2025.0, "team_name": "Borussia Dortmund", "next_count": 5.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--d0baaf1b-c29d-45a7-8840-269ab144e44b-0', tool_calls=[{'name': 'get_next_fixtures', 'args': {'league_name': 'Club World Cup', 'season': 2025.0, 'team_name': 'Borussia Dortmund', 'next_count': 5.0}, 'id': '2a397e57-f942-4006-93ad-92d5cbfc9d43', 'type': 'tool_call'}], usage_metadata={'input_tokens': 187, 'output_tokens': 24, 'total_tokens': 211, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="data=

## 3. Last fixtures

<https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `last` parameter

In [55]:
def get_last_fixtures(
    league_name: str | None = None,
    season: int | None = None,
    team_name: str | None = None,
    last_count: int = 5,
    date_from: str | None = None,
    date_to: str | None = None
) -> Response:
    """
    Get the last fixtures for a league, team, or date range.
    
    Args:
        league_name: Name of the league or cup (e.g., "Premier League", "La Liga")
        season: Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)

        team_name: Optional team name to filter fixtures for specific team
        last_count: Optional number of last fixtures to retrieve (max 20)
        date_from: Optional start date in YYYY-MM-DD format
        date_to: Optional end date in YYYY-MM-DD format
        
    Returns:
        ValidResponse with fixtures data or ErrorResponse with error details
    """
    params = {}
    
    # Add league parameter if provided
    if league_name:
        league_id = get_league_id_by_name(league_name)
        if league_id is None:
            logger.error(f"League '{league_name}' not found")
            return ErrorResponse(error=f"League '{league_name}' not found")
        params["league"] = league_id
    
    # Add season parameter if provided
    if season:
        params["season"] = season
    
    # Add team parameter if provided
    if team_name:
        team_id = get_team_id_by_name(team_name)
        if team_id is None:
            logger.error(f"Team '{team_name}' not found")
            return ErrorResponse(error=f"Team '{team_name}' not found")
        params["team"] = team_id
    
    # Add last parameter (number of last fixtures)
    if last_count and last_count > 0:
        params["last"] = min(last_count, 20)  # API limit is 20
    
    # Add date range parameters if provided
    if date_from:
        params["from"] = date_from
    if date_to:
        params["to"] = date_to
    
    logger.info(f"Fetching last {last_count} fixtures")
    if league_name:
        logger.info(f"League: {league_name}")
    if team_name:
        logger.info(f"Team: {team_name}")
    
    # Make the API call
    try:
        response = call_football_api("GET", "fixtures", params=params)
        
        if isinstance(response, ValidResponse) and "response" in response.data:
            logger.info(f"Successfully retrieved {len(response.data['response'])} fixtures")
            return response
        elif isinstance(response, ValidResponse):
            logger.warning(f"No fixtures found")
            return ErrorResponse(error="No fixtures found")
        else:
            return response  # Already an ErrorResponse
            
    except Exception as e:
        logger.error(f"Error fetching fixtures: {str(e)}")
        return ErrorResponse(error=f"Error fetching fixtures: {str(e)}")

In [58]:
# Test
fixtures = get_last_fixtures(league_name="Club World Cup", season=2025, team_name="Porto", last_count=5)
print("Raw API Response:")
print(fixtures)

INFO:__main__:Fetching last 5 fixtures
INFO:__main__:League: Club World Cup
INFO:__main__:Team: Porto
INFO:__main__:Fetching last 5 fixtures
INFO:__main__:League: Club World Cup
INFO:__main__:Team: Porto
INFO:__main__:Successfully retrieved 2 fixtures
INFO:__main__:Successfully retrieved 2 fixtures


Raw API Response:
data={'get': 'fixtures', 'parameters': {'league': '15', 'season': '2025', 'team': '212', 'last': '5'}, 'errors': [], 'results': 2, 'paging': {'current': 1, 'total': 1}, 'response': [{'fixture': {'id': 1321697, 'referee': 'Cristian Garay, Chile', 'timezone': 'UTC', 'date': '2025-06-19T19:00:00+00:00', 'timestamp': 1750359600, 'periods': {'first': 1750359600, 'second': 1750363200}, 'venue': {'id': 1898, 'name': 'Mercedes-Benz Stadium', 'city': 'Atlanta, Georgia'}, 'status': {'long': 'Match Finished', 'short': 'FT', 'elapsed': 90, 'extra': 8}}, 'league': {'id': 15, 'name': 'FIFA Club World Cup', 'country': 'World', 'logo': 'https://media.api-sports.io/football/leagues/15.png', 'flag': None, 'season': 2025, 'round': 'Group Stage - 2', 'standings': True}, 'teams': {'home': {'id': 9568, 'name': 'Inter Miami', 'logo': 'https://media.api-sports.io/football/teams/9568.png', 'winner': True}, 'away': {'id': 212, 'name': 'FC Porto', 'logo': 'https://media.api-sports.io/football/t

In [56]:
class GetLastFixturesInput(BaseModel):
    league_name: Optional[str] = Field(None, description="Name of the league or cup (e.g., 'Premier League', 'La Liga')")
    season: Optional[int] = Field(None, description="Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)")
    team_name: Optional[str] = Field(None, description="Optional team name to filter fixtures for specific team")
    last_count: int = Field(5, description="Number of last fixtures to retrieve (max 20)")
    date_from: Optional[str] = Field(None, description="Optional start date in YYYY-MM-DD format")
    date_to: Optional[str] = Field(None, description="Optional end date in YYYY-MM-DD format")

# create a tool from the get_last_fixtures function
get_last_fixtures_tool = StructuredTool.from_function(
    get_last_fixtures,
    name="get_last_fixtures",
    description="Get the last fixtures for a league, team, or date range.",
    args_schema=GetLastFixturesInput,
    return_direct=False
)

print(get_last_fixtures_tool.name)
print(get_last_fixtures_tool.description)
print(get_last_fixtures_tool.args)

get_last_fixtures
Get the last fixtures for a league, team, or date range.
{'league_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': "Name of the league or cup (e.g., 'Premier League', 'La Liga')", 'title': 'League Name'}, 'season': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Season year (4 digits, e.g., 2024)', 'title': 'Season'}, 'team_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional team name to filter fixtures for specific team', 'title': 'Team Name'}, 'last_count': {'default': 5, 'description': 'Number of last fixtures to retrieve (max 20)', 'title': 'Last Count', 'type': 'integer'}, 'date_from': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional start date in YYYY-MM-DD format', 'title': 'Date From'}, 'date_to': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional end date 

In [61]:
react_agent = create_react_agent(
    model=model,
    tools=[get_last_fixtures_tool],
    prompt=f"You are a helpful assistant. Today is {datetime.today().strftime('%Y-%m-%d')}. If the user query does not specify a season, assume it is 2024 for European leagues and 2025 for cups."
)

react_agent.invoke(
    {"messages": [{"role": "user", "content": "what were the last 5 games played by Chelsea?"}]}
)

INFO:__main__:Fetching last 5 fixtures
INFO:__main__:Team: Chelsea
INFO:__main__:Team: Chelsea
INFO:__main__:Successfully retrieved 5 fixtures
INFO:__main__:Successfully retrieved 5 fixtures


{'messages': [HumanMessage(content='what were the last 5 games played by Chelsea?', additional_kwargs={}, response_metadata={}, id='2866547f-8baa-4f56-90fd-da66f12fa84a'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_last_fixtures', 'arguments': '{"last_count": 5.0, "team_name": "Chelsea"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--c54f57f4-adfb-4537-bdae-716bfc9ee9b0-0', tool_calls=[{'name': 'get_last_fixtures', 'args': {'last_count': 5.0, 'team_name': 'Chelsea'}, 'id': '83790fcf-ecc1-4245-a31b-0690cfa3d006', 'type': 'tool_call'}], usage_metadata={'input_tokens': 184, 'output_tokens': 13, 'total_tokens': 197, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="data={'get': 'fixtures', 'parameters': {'team': '49', 'last': '5'}, 'errors': [], 'results': 5, 'paging': {'current': 1, 'total': 1}, 'response': [{'fixt

In [ ]:
"""
## 4. Specific fixtures
"""

In [103]:
def get_specific_fixture(
    league_name: str,
    season: int,
    home_team: str,
    away_team: str
) -> Response:
    """
    Get a specific fixture between two teams in a given league and season.
    
    Args:
        league_name: Name of the league or cup (e.g., "Premier League", "La Liga")
        season: Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)

        home_team: Name of the home team
        away_team: Name of the away team
        
    Returns:
        ValidResponse with fixture data or ErrorResponse with error details
    """
    # Get league ID
    league_id = get_league_id_by_name(league_name)
    if league_id is None:
        logger.error(f"League '{league_name}' not found")
        return ErrorResponse(error=f"League '{league_name}' not found")
    
    # Get team IDs
    home_team_id = get_team_id_by_name(home_team)
    if home_team_id is None:
        logger.error(f"Home team '{home_team}' not found")
        return ErrorResponse(error=f"Home team '{home_team}' not found")
    
    away_team_id = get_team_id_by_name(away_team)
    if away_team_id is None:
        logger.error(f"Away team '{away_team}' not found")
        return ErrorResponse(error=f"Away team '{away_team}' not found")
    
    logger.info(f"Searching for fixture: {home_team} vs {away_team} in {league_name} season {season}")
    
    # First, try to get all fixtures for the home team in that league/season
    params = {
        "league": league_id,
        "season": season,
        "team": home_team_id
    }
    
    try:
        response = call_football_api("GET", "fixtures", params=params)
        
        if isinstance(response, ValidResponse) and "response" in response.data:
            fixtures = response.data["response"]
            
            # Filter for matches against the away team
            matching_fixtures = []
            for fixture in fixtures:
                home_id = fixture["teams"]["home"]["id"]
                away_id = fixture["teams"]["away"]["id"]
                
                # Check if this is the specific match we're looking for
                if home_id == home_team_id and away_id == away_team_id:
                    matching_fixtures.append(fixture)
            
            if matching_fixtures:
                logger.info(f"Found {len(matching_fixtures)} fixture(s) between {home_team} and {away_team}")
                # Return the fixtures in the same format as the API
                return ValidResponse(data={"response": matching_fixtures})
            else:
                # Try the reverse (away team as home team parameter)
                params["team"] = away_team_id
                response2 = call_football_api("GET", "fixtures", params=params)
                
                if isinstance(response2, ValidResponse) and "response" in response2.data:
                    fixtures2 = response2.data["response"]
                    
                    # Filter for matches against the home team (but with teams swapped)
                    for fixture in fixtures2:
                        home_id = fixture["teams"]["home"]["id"]
                        away_id = fixture["teams"]["away"]["id"]
                        
                        if home_id == home_team_id and away_id == away_team_id:
                            matching_fixtures.append(fixture)
                
                if matching_fixtures:
                    logger.info(f"Found {len(matching_fixtures)} fixture(s) between {home_team} and {away_team}")
                    return ValidResponse(data={"response": matching_fixtures})
                else:
                    error_msg = f"No fixture found between {home_team} (home) and {away_team} (away) in {league_name} season {season}"
                    logger.warning(error_msg)
                    return ErrorResponse(error=error_msg)
        else:
            return response  # Already an ErrorResponse
            
    except Exception as e:
        logger.error(f"Error searching for fixture: {str(e)}")
        return ErrorResponse(error=f"Error searching for fixture: {str(e)}")

In [104]:
class GetSpecificFixtureInput(BaseModel):
    league_name: str = Field(..., description="Name of the league or cup (e.g., 'Premier League', 'La Liga')")
    season: int = Field(..., description="Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)")
    home_team: str = Field(..., description="Name of the home team")
    away_team: str = Field(..., description="Name of the away team")

# create a tool from the get_specific_fixture function
get_specific_fixture_tool = StructuredTool.from_function(
    get_specific_fixture,
    name="get_specific_fixture",
    description="Get a specific fixture between two teams in a given league and season.",
    args_schema=GetSpecificFixtureInput,
    return_direct=False
)

print(get_specific_fixture_tool.name)
print(get_specific_fixture_tool.description)
print(get_specific_fixture_tool.args)

get_specific_fixture
Get a specific fixture between two teams in a given league and season.
{'league_name': {'description': "Name of the league or cup (e.g., 'Premier League', 'La Liga')", 'title': 'League Name', 'type': 'string'}, 'season': {'description': 'Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)', 'title': 'Season', 'type': 'integer'}, 'home_team': {'description': 'Name of the home team', 'title': 'Home Team', 'type': 'string'}, 'away_team': {'description': 'Name of the away team', 'title': 'Away Team', 'type': 'string'}}


In [100]:
# Test the specific fixture function
test_fixture = get_specific_fixture("Premier League", 2024, "Arsenal", "Chelsea")
print("Specific fixture search result:")
print(test_fixture)

INFO:__main__:Searching for fixture: Arsenal vs Chelsea in Premier League season 2024


INFO:__main__:Found 1 fixture(s) between Arsenal and Chelsea


Specific fixture search result:
data={'response': [{'fixture': {'id': 1208304, 'referee': 'C. Kavanagh', 'timezone': 'UTC', 'date': '2025-03-16T13:30:00+00:00', 'timestamp': 1742131800, 'periods': {'first': 1742131800, 'second': 1742135400}, 'venue': {'id': 494, 'name': 'Emirates Stadium', 'city': 'London'}, 'status': {'long': 'Match Finished', 'short': 'FT', 'elapsed': 90, 'extra': 5}}, 'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2024, 'round': 'Regular Season - 29', 'standings': True}, 'teams': {'home': {'id': 42, 'name': 'Arsenal', 'logo': 'https://media.api-sports.io/football/teams/42.png', 'winner': True}, 'away': {'id': 49, 'name': 'Chelsea', 'logo': 'https://media.api-sports.io/football/teams/49.png', 'winner': False}}, 'goals': {'home': 1, 'away': 0}, 'score': {'halftime': {'home': 1, 'away': 0}, 'fulltime': {'home': 1, 'away'

In [106]:
react_agent = create_react_agent(
    model=model,
    tools=[get_specific_fixture_tool],
    prompt=f"You are a helpful assistant. Today is {datetime.today().strftime('%Y-%m-%d')}. If the user query does not specify a season, assume it is 2024 for European leagues and 2025 for cups. If the user does not specify a league, assume it is the national league for the teams in the query. If any tool fails, figure out the correct parameters (e.g. for team name, 'FCP' should be 'FC Porto') and try again."
)

react_agent.invoke(
    {"messages": [{"role": "user", "content": "What can you tell me about SLB vs SCP in the league in 2012/2013?"}]}
)

ERROR:__main__:Home team 'SLB' not found
INFO:__main__:Searching for fixture: Benfica vs Sporting CP in Primeira Liga season 2012
INFO:__main__:Found 1 fixture(s) between Benfica and Sporting CP


{'messages': [HumanMessage(content='What can you tell me about SLB vs SCP in the league in 2012/2013?', additional_kwargs={}, response_metadata={}, id='7589365d-a4cb-4aba-a6ff-62b65869d155'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_specific_fixture', 'arguments': '{"league_name": "Primeira Liga", "away_team": "SCP", "season": 2012.0, "home_team": "SLB"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--9e943c1b-22d3-4c97-95ae-1694e9352b6c-0', tool_calls=[{'name': 'get_specific_fixture', 'args': {'league_name': 'Primeira Liga', 'away_team': 'SCP', 'season': 2012.0, 'home_team': 'SLB'}, 'id': 'c77a803e-1376-4c39-9a0b-5d004cfe0c48', 'type': 'tool_call'}], usage_metadata={'input_tokens': 232, 'output_tokens': 22, 'total_tokens': 254, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='error="Home team \'SLB\' not foun

## 5. Match results

Already covered by the specific fixtures tool.

## 6. Match events

TODO: create helper get_player_id_by_name to filter by player id? (might not be necessary)

In [113]:
def get_match_events(
    fixture_id: int,
    team_name: str | None = None,
    player_name: str | None = None,
    event_type: str | None = None
) -> Response:
    """
    Get the events from a fixture.
    
    Available event types:
    - Goal: Normal Goal, Own Goal, Penalty, Missed Penalty
    - Card: Yellow Card, Red Card
    - Subst: Substitution [1, 2, 3...]
    - VAR: Goal cancelled, Penalty confirmed (available from 2020-2021 season)
    
    Args:
        fixture_id: The ID of the fixture (required)
        team_name: Optional team name to filter events by team
        player_name: Optional player name to filter events by player
        event_type: Optional event type to filter by (Goal, Card, Subst, VAR)
        
    Returns:
        ValidResponse with events data or ErrorResponse with error details
    """
    # Prepare parameters for the API call
    params: dict[str, int | str] = {
        "fixture": fixture_id
    }
    
    # Add team parameter if provided
    if team_name:
        team_id = get_team_id_by_name(team_name)
        if team_id is None:
            logger.error(f"Team '{team_name}' not found")
            return ErrorResponse(error=f"Team '{team_name}' not found")
        params["team"] = team_id
    
    # Add event type parameter if provided
    if event_type:
        # Validate event type
        valid_types = ["Goal", "Card", "Subst", "VAR"]
        if event_type not in valid_types:
            logger.error(f"Invalid event type '{event_type}'. Valid types: {valid_types}")
            return ErrorResponse(error=f"Invalid event type '{event_type}'. Valid types: {valid_types}")
        params["type"] = event_type
    
    logger.info(f"Fetching events for fixture {fixture_id}")
    if team_name:
        logger.info(f"Filtering for team: {team_name}")
    if event_type:
        logger.info(f"Filtering for event type: {event_type}")
    
    # Make the API call
    try:
        response = call_football_api("GET", "fixtures/events", params=params)
        
        if isinstance(response, ValidResponse) and "response" in response.data:
            events = response.data["response"]
            
            # Additional filtering by player name if provided (API doesn't support this directly)
            if player_name and events:
                filtered_events = []
                for event in events:
                    player = event.get("player", {})
                    if player and player.get("name"):
                        if player_name.lower() in player["name"].lower() or player["name"].lower() in player_name.lower():
                            filtered_events.append(event)
                
                logger.info(f"Filtered {len(filtered_events)} events for player '{player_name}'")
                return ValidResponse(data={"response": filtered_events})
            
            logger.info(f"Successfully retrieved {len(events)} events")
            return response
        elif isinstance(response, ValidResponse):
            logger.warning(f"No events found for fixture {fixture_id}")
            return ErrorResponse(error=f"No events found for fixture {fixture_id}")
        else:
            return response  # Already an ErrorResponse
            
    except Exception as e:
        logger.error(f"Error fetching match events: {str(e)}")
        return ErrorResponse(error=f"Error fetching match events: {str(e)}")



In [ ]:
# Test the match events function
# First, let's get a specific fixture to test with
test_fixture_response = get_specific_fixture("Premier League", 2024, "Arsenal", "Chelsea")

if isinstance(test_fixture_response, ValidResponse) and test_fixture_response.data["response"]:
    fixture = test_fixture_response.data["response"][0]
    fixture_id = fixture["fixture"]["id"]
    
    print(f"Testing match events for fixture ID: {fixture_id}")
    print(f"Match: {fixture['teams']['home']['name']} vs {fixture['teams']['away']['name']}")
    
    # Test the events function
    events_response = get_match_events(fixture_id)
    print("\nRaw events response:")
    print(events_response)
    
else:
    print("Could not find a test fixture. Let's test with a known fixture ID.")
    # Test with example fixture ID from the API documentation
    test_events = get_match_events(215662)
    print("Test events response:")
    print(test_events)

INFO:__main__:Searching for fixture: Arsenal vs Chelsea in Premier League season 2024


INFO:__main__:Found 1 fixture(s) between Arsenal and Chelsea
INFO:__main__:Fetching events for fixture 1208304
INFO:__main__:Fetching events for fixture 1208304
INFO:__main__:Successfully retrieved 14 events
INFO:__main__:Successfully retrieved 14 events


Testing match events for fixture ID: 1208304
Match: Arsenal vs Chelsea

Raw events response:
data={'get': 'fixtures/events', 'parameters': {'fixture': '1208304'}, 'errors': [], 'results': 14, 'paging': {'current': 1, 'total': 1}, 'response': [{'time': {'elapsed': 20, 'extra': None}, 'team': {'id': 42, 'name': 'Arsenal', 'logo': 'https://media.api-sports.io/football/teams/42.png'}, 'player': {'id': 47311, 'name': 'Mikel Merino'}, 'assist': {'id': 37127, 'name': 'M. Ødegaard'}, 'type': 'Goal', 'detail': 'Normal Goal', 'comments': None}, {'time': {'elapsed': 30, 'extra': None}, 'team': {'id': 49, 'name': 'Chelsea', 'logo': 'https://media.api-sports.io/football/teams/49.png'}, 'player': {'id': 152953, 'name': 'Levi Colwill'}, 'assist': {'id': None, 'name': None}, 'type': 'Card', 'detail': 'Yellow Card', 'comments': 'Foul'}, {'time': {'elapsed': 52, 'extra': None}, 'team': {'id': 49, 'name': 'Chelsea', 'logo': 'https://media.api-sports.io/football/teams/49.png'}, 'player': {'id': 1864, 'nam

In [ ]:

def get_match_events_by_teams(
    league_name: str,
    season: int,
    home_team: str,
    away_team: str,
    team_name: str | None = None,
    player_name: str | None = None,
    event_type: str | None = None
) -> Response:
    """
    Get the events from a match between two specific teams.
    
    This is a wrapper function that first finds the fixture between the teams,
    then retrieves the events for that fixture.
    
    Available event types:
    - Goal: Normal Goal, Own Goal, Penalty, Missed Penalty
    - Card: Yellow Card, Red Card
    - Subst: Substitution [1, 2, 3...]
    - VAR: Goal cancelled, Penalty confirmed (available from 2020-2021 season)
    
    Args:
        league_name: Name of the league or cup (e.g., "Premier League", "La Liga")
        season: Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)
        home_team: Name of the home team
        away_team: Name of the away team
        team_name: Optional team name to filter events by team
        player_name: Optional player name to filter events by player
        event_type: Optional event type to filter by (Goal, Card, Subst, VAR)
        
    Returns:
        ValidResponse with events data or ErrorResponse with error details
    """
    # First, find the fixture between the two teams
    logger.info(f"Looking for fixture: {home_team} vs {away_team} in {league_name} season {season}")
    
    fixture_response = get_specific_fixture(league_name, season, home_team, away_team)
    
    if isinstance(fixture_response, ErrorResponse):
        return fixture_response
    
    if not isinstance(fixture_response, ValidResponse) or not fixture_response.data.get("response"):
        return ErrorResponse(error=f"No fixture found between {home_team} and {away_team} in {league_name} season {season}")
    
    fixtures = fixture_response.data["response"]
    
    # Use the first fixture if multiple matches found
    if len(fixtures) > 1:
        logger.info(f"Found {len(fixtures)} fixtures between these teams. Using the first one.")
    
    fixture = fixtures[0]
    fixture_id = fixture["fixture"]["id"]
    
    logger.info(f"Found fixture ID {fixture_id}, now getting events...")
    
    # Now get the events for this fixture
    return get_match_events(fixture_id, team_name, player_name, event_type)

In [119]:
# Test with a known match
test_events_wrapper = get_match_events_by_teams(
    league_name="Primeira Liga", 
    season=2012, 
    home_team="Benfica", 
    away_team="Sporting CP"
)

print("\nWrapper function result:")
print(test_events_wrapper)

INFO:__main__:Looking for fixture: Benfica vs Sporting CP in Primeira Liga season 2012
INFO:__main__:Searching for fixture: Benfica vs Sporting CP in Primeira Liga season 2012
INFO:__main__:Found 1 fixture(s) between Benfica and Sporting CP
INFO:__main__:Found fixture ID 202823, now getting events...
INFO:__main__:Fetching events for fixture 202823
INFO:__main__:Successfully retrieved 8 events



Wrapper function result:
data={'get': 'fixtures/events', 'parameters': {'fixture': '202823'}, 'errors': [], 'results': 8, 'paging': {'current': 1, 'total': 1}, 'response': [{'time': {'elapsed': 36, 'extra': None}, 'team': {'id': 211, 'name': 'Benfica', 'logo': 'https://media.api-sports.io/football/teams/211.png'}, 'player': {'id': 576, 'name': 'E. Salvio'}, 'assist': {'id': None, 'name': None}, 'type': 'Goal', 'detail': 'Normal Goal', 'comments': None}, {'time': {'elapsed': 65, 'extra': None}, 'team': {'id': 228, 'name': 'Sporting CP', 'logo': 'https://media.api-sports.io/football/teams/228.png'}, 'player': {'id': 175, 'name': 'E. Dier'}, 'assist': {'id': 36979, 'name': 'S. Schaars'}, 'type': 'subst', 'detail': 'Substitution 1', 'comments': None}, {'time': {'elapsed': 65, 'extra': None}, 'team': {'id': 228, 'name': 'Sporting CP', 'logo': 'https://media.api-sports.io/football/teams/228.png'}, 'player': {'id': 47028, 'name': 'Capel'}, 'assist': {'id': 64173, 'name': 'V. Viola'}, 'type':

In [121]:
class GetMatchEventsByTeamsInput(BaseModel):
    league_name: str = Field(..., description="Name of the league or cup (e.g., 'Premier League', 'La Liga')")
    season: int = Field(..., description="Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)")
    home_team: str = Field(..., description="Name of the home team")
    away_team: str = Field(..., description="Name of the away team")
    team_name: Optional[str] = Field(None, description="Optional team name to filter events by team")
    player_name: Optional[str] = Field(None, description="Optional player name to filter events by player")
    event_type: Optional[str] = Field(None, description="Optional event type to filter by (Goal, Card, Subst, VAR)")

# create a tool from the get_match_events_by_teams function
get_match_events_by_teams_tool = StructuredTool.from_function(
    get_match_events_by_teams,
    name="get_match_events_by_teams",
    description="Get the events from a match between two specific teams (automatically finds the fixture and retrieves events).",
    args_schema=GetMatchEventsByTeamsInput,
    return_direct=False
)

print(get_match_events_by_teams_tool.name)
print(get_match_events_by_teams_tool.description)
print(get_match_events_by_teams_tool.args)

get_match_events_by_teams
Get the events from a match between two specific teams (automatically finds the fixture and retrieves events).
{'league_name': {'description': "Name of the league or cup (e.g., 'Premier League', 'La Liga')", 'title': 'League Name', 'type': 'string'}, 'season': {'description': 'Season FIRST year (4 digits, e.g., 2024 for 2024/2025 season)', 'title': 'Season', 'type': 'integer'}, 'home_team': {'description': 'Name of the home team', 'title': 'Home Team', 'type': 'string'}, 'away_team': {'description': 'Name of the away team', 'title': 'Away Team', 'type': 'string'}, 'team_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional team name to filter events by team', 'title': 'Team Name'}, 'player_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional player name to filter events by player', 'title': 'Player Name'}, 'event_type': {'anyOf': [{'type': 'string'}, {'type': 'null'}], '

In [122]:
# Test the new wrapper function that combines fixture lookup and events retrieval
react_agent_events = create_react_agent(
    model=model,
    tools=[get_match_events_by_teams_tool],
    prompt=f"You are a helpful football assistant. Today is {datetime.today().strftime('%Y-%m-%d')}. If the user query does not specify a season, assume it is 2024 for European leagues and 2025 for cups. If the user does not specify a league, assume it is the national league for the teams in the query. If any tool fails, figure out the correct parameters (e.g. for team name, 'FCP' should be 'FC Porto') and try again."
)

print("Testing match events with ReAct agent using the wrapper function:")
result = react_agent_events.invoke({
    "messages": [{"role": "user", "content": "What were the events (goals, cards, substitutions) in the SLB vs SCP match in 2012/2013?"}]
})

print("\nAgent result:")
print(result)

Testing match events with ReAct agent using the wrapper function:


INFO:__main__:Looking for fixture: SLB vs SCP in Primeira Liga season 2012
ERROR:__main__:Home team 'SLB' not found
ERROR:__main__:Home team 'SLB' not found
INFO:__main__:Looking for fixture: Benfica vs Sporting CP in Primeira Liga season 2012
INFO:__main__:Searching for fixture: Benfica vs Sporting CP in Primeira Liga season 2012
INFO:__main__:Looking for fixture: Benfica vs Sporting CP in Primeira Liga season 2012
INFO:__main__:Searching for fixture: Benfica vs Sporting CP in Primeira Liga season 2012
INFO:__main__:Found 1 fixture(s) between Benfica and Sporting CP
INFO:__main__:Found fixture ID 202823, now getting events...
INFO:__main__:Fetching events for fixture 202823
INFO:__main__:Found 1 fixture(s) between Benfica and Sporting CP
INFO:__main__:Found fixture ID 202823, now getting events...
INFO:__main__:Fetching events for fixture 202823
INFO:__main__:Successfully retrieved 8 events
INFO:__main__:Successfully retrieved 8 events



Agent result:
{'messages': [HumanMessage(content='What were the events (goals, cards, substitutions) in the SLB vs SCP match in 2012/2013?', additional_kwargs={}, response_metadata={}, id='0bae6fb1-ad7d-48ed-b720-c720ea43fcdd'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_match_events_by_teams', 'arguments': '{"league_name": "Primeira Liga", "away_team": "SCP", "season": 2012.0, "home_team": "SLB"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--a2902563-9149-46c4-a8a2-03f69049bafa-0', tool_calls=[{'name': 'get_match_events_by_teams', 'args': {'league_name': 'Primeira Liga', 'away_team': 'SCP', 'season': 2012.0, 'home_team': 'SLB'}, 'id': '73914c5e-74c0-4ec7-afb6-e1fbf9afe80b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 294, 'output_tokens': 26, 'total_tokens': 320, 'input_token_details': {'cache_read': 0}}), ToolMessag

## 7. Player statistics

## Extra

### 8. Odds

### 9. Head-to-head (H2H)